# Washington State Dental Loss Ratio Insurance Analytics

This notebook analyzes Washington State dental insurer loss ratio data using Python.

**Project goals:**
- Clean and prepare insurer-level dental loss ratio data.
- Analyze premiums, payments, members, member months, and dental loss ratios.
- Compare data-loading performance using Pandas and Dask.
- Build regression and classification models.
- Identify important predictors for insurance business outcomes.
- Generate output files that can be used in a dashboard.

The project aligns with the presentation topic: **Analysis of Washington State Insurer's Dental Loss Ratios**.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import cProfile
import pstats
import io
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (9, 5)
sns.set_theme(style="whitegrid")


## 2. Optional: Install and Import Dask

Dask is useful for scaling data processing workflows. If Dask is not installed in your environment, run the install cell below.


In [ ]:
# Run this only if Dask is not installed:
# !pip install dask[dataframe] --quiet

try:
    import dask.dataframe as dd
    DASK_AVAILABLE = True
    print("Dask is available.")
except Exception as e:
    DASK_AVAILABLE = False
    print("Dask is not available. Pandas analysis will still run.")


## 3. Upload and Load Dataset

Upload the Washington State Dental Loss Ratios CSV file when prompted.

Expected dataset fields may include:
- Company Name
- NAIC Code
- Domicile State
- Business Type
- Year
- Dental Premiums
- Dental Payments
- Dental Members
- Dental Member Months
- Dental Loss Ratio
- Average Amount of Premiums per Member per Month
- Previous Year Average Amount of Premiums per Member per Month
- Percentage Change in Average Premium per Member per Month from Previous Year


In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


In [ ]:
csv_files = [file for file in uploaded.keys() if file.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload at least one CSV file.")

file_path = csv_files[0]

start_time = time.time()
df = pd.read_csv(file_path)
pandas_load_time = time.time() - start_time

print("Loaded file:", file_path)
print("Pandas load time:", round(pandas_load_time, 4), "seconds")
print("Shape:", df.shape)

df.head()


## 4. Compare Pandas vs Dask Loading Time

In [ ]:
if DASK_AVAILABLE:
    start_time = time.time()
    dask_df = dd.read_csv(file_path, assume_missing=True)
    # Trigger computation with a small operation
    dask_shape = (len(dask_df), len(dask_df.columns))
    dask_load_time = time.time() - start_time

    print("Dask load time:", round(dask_load_time, 4), "seconds")
    print("Dask shape:", dask_shape)
else:
    dask_load_time = np.nan
    print("Skipping Dask comparison because Dask is unavailable.")

performance_loading = pd.DataFrame({
    "Method": ["Pandas read_csv", "Dask read_csv"],
    "Loading Time Seconds": [pandas_load_time, dask_load_time]
})

performance_loading


## 5. Data Cleaning and Column Standardization

In [ ]:
# Create working copy
data = df.copy()

# Standardize column names
data.columns = (
    data.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
    .str.replace("__", "_")
)

print("Standardized columns:")
print(data.columns.tolist())


In [ ]:
# Helper function to find columns flexibly

def find_column(possible_names, columns):
    normalized = {col.lower(): col for col in columns}
    for name in possible_names:
        key = name.lower()
        if key in normalized:
            return normalized[key]
    return None

company_col = find_column(["Company_Name", "Company"], data.columns)
naic_col = find_column(["NAIC_Code", "NAIC"], data.columns)
domicile_col = find_column(["Domicile_State", "Domicile"], data.columns)
business_col = find_column(["Business_Type", "Business"], data.columns)
year_col = find_column(["Year"], data.columns)

premium_col = find_column(["Dental_Premiums", "Premiums"], data.columns)
payment_col = find_column(["Dental_Payments", "Payments"], data.columns)
members_col = find_column(["Dental_Members", "Members"], data.columns)
member_months_col = find_column(["Dental_Member_Months", "Member_Months"], data.columns)
loss_ratio_col = find_column(["Dental_Loss_Ratio", "Loss_Ratio"], data.columns)
avg_premium_col = find_column([
    "Average_Amount_of_Premiums_per_Member_per_Month",
    "Average_Premiums_per_Member_per_Month",
    "Average_Amount_of_Premium_per_Member_per_Month"
], data.columns)
prev_avg_premium_col = find_column([
    "Previous_Year_Average_Amount_of_Premiums_per_Member_per_Month",
    "Previous_Year_Average_Premium_per_Member_per_Month"
], data.columns)
pct_change_col = find_column([
    "Percentage_Change_in_Average_Premium_per_Member_per_Month_from_Previous_Year",
    "Percentage_Change_in_Average_Premium"
], data.columns)

important_cols = {
    "company_col": company_col,
    "naic_col": naic_col,
    "domicile_col": domicile_col,
    "business_col": business_col,
    "year_col": year_col,
    "premium_col": premium_col,
    "payment_col": payment_col,
    "members_col": members_col,
    "member_months_col": member_months_col,
    "loss_ratio_col": loss_ratio_col,
    "avg_premium_col": avg_premium_col,
    "prev_avg_premium_col": prev_avg_premium_col,
    "pct_change_col": pct_change_col
}

important_cols


In [ ]:
# Clean numeric columns by removing symbols and commas

numeric_candidates = [
    premium_col, payment_col, members_col, member_months_col,
    loss_ratio_col, avg_premium_col, prev_avg_premium_col, pct_change_col, year_col
]

for col in numeric_candidates:
    if col and col in data.columns:
        data[col] = (
            data[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        data[col] = pd.to_numeric(data[col], errors="coerce")

# Clean categorical columns
categorical_candidates = [company_col, domicile_col, business_col]

for col in categorical_candidates:
    if col and col in data.columns:
        data[col] = data[col].astype(str).str.strip()

print("Missing values:")
data.isnull().sum().sort_values(ascending=False).head(15)


## 6. Data Overview

In [ ]:
print("Rows and columns:", data.shape)
print("\nYears available:")
if year_col:
    print(sorted(data[year_col].dropna().unique()))
else:
    print("Year column not found.")

print("\nNumber of companies:")
if company_col:
    print(data[company_col].nunique())
else:
    print("Company column not found.")

print("\nBusiness types:")
if business_col:
    print(data[business_col].value_counts(dropna=False))
else:
    print("Business Type column not found.")


## 7. Exploratory Data Analysis

In [ ]:
# Dental loss ratio distribution

if loss_ratio_col:
    sns.histplot(data[loss_ratio_col].dropna(), bins=25, kde=True)
    plt.title("Distribution of Dental Loss Ratio")
    plt.xlabel("Dental Loss Ratio")
    plt.ylabel("Count")
    plt.show()
else:
    print("Dental Loss Ratio column not found.")


In [ ]:
# Dental premiums by year

if year_col and premium_col:
    yearly_premiums = data.groupby(year_col)[premium_col].sum().reset_index()

    sns.lineplot(data=yearly_premiums, x=year_col, y=premium_col, marker="o")
    plt.title("Total Dental Premiums by Year")
    plt.xlabel("Year")
    plt.ylabel("Total Dental Premiums")
    plt.show()
else:
    print("Year and/or Dental Premiums column not found.")


In [ ]:
# Dental payments by year

if year_col and payment_col:
    yearly_payments = data.groupby(year_col)[payment_col].sum().reset_index()

    sns.lineplot(data=yearly_payments, x=year_col, y=payment_col, marker="o")
    plt.title("Total Dental Payments by Year")
    plt.xlabel("Year")
    plt.ylabel("Total Dental Payments")
    plt.show()
else:
    print("Year and/or Dental Payments column not found.")


In [ ]:
# Average loss ratio by year

if year_col and loss_ratio_col:
    yearly_loss_ratio = data.groupby(year_col)[loss_ratio_col].mean().reset_index()

    sns.lineplot(data=yearly_loss_ratio, x=year_col, y=loss_ratio_col, marker="o")
    plt.title("Average Dental Loss Ratio by Year")
    plt.xlabel("Year")
    plt.ylabel("Average Dental Loss Ratio")
    plt.show()
else:
    print("Year and/or Dental Loss Ratio column not found.")


In [ ]:
# Top companies by dental premiums

if company_col and premium_col:
    top_companies = (
        data.groupby(company_col)[premium_col]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    sns.barplot(data=top_companies, x=premium_col, y=company_col)
    plt.title("Top 10 Companies by Dental Premiums")
    plt.xlabel("Total Dental Premiums")
    plt.ylabel("Company")
    plt.show()
else:
    print("Company and/or Dental Premiums column not found.")


In [ ]:
# Dental payments vs dental premiums

if premium_col and payment_col:
    sns.scatterplot(data=data, x=premium_col, y=payment_col, hue=business_col if business_col else None)
    plt.title("Dental Payments vs Dental Premiums")
    plt.xlabel("Dental Premiums")
    plt.ylabel("Dental Payments")
    plt.show()
else:
    print("Premium and/or Payment column not found.")


In [ ]:
# Correlation heatmap for numeric columns

numeric_data = data.select_dtypes(include=["int64", "float64"])

if numeric_data.shape[1] > 1:
    corr = numeric_data.corr()
    sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f")
    plt.title("Correlation Heatmap")
    plt.show()
else:
    print("Not enough numeric columns for correlation heatmap.")


## 8. Regression: Predicting Dental Loss Ratio

In [ ]:
# Select available regression features

regression_features = [
    member_months_col,
    members_col,
    payment_col,
    premium_col,
    avg_premium_col
]

regression_features = [col for col in regression_features if col and col in data.columns]

if not loss_ratio_col:
    raise ValueError("Dental Loss Ratio column not found. Cannot run regression.")

regression_df = data[regression_features + [loss_ratio_col]].copy()
regression_df = regression_df.dropna(subset=[loss_ratio_col])

print("Regression features:", regression_features)
print("Regression dataset shape:", regression_df.shape)


In [ ]:
X_reg = regression_df[regression_features]
y_reg = regression_df[loss_ratio_col]

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_reg_imputed = imputer.fit_transform(X_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg_imputed, y_reg, test_size=0.30, random_state=42
)

start_time = time.time()
linear_model = LinearRegression()
linear_model.fit(X_train_reg, y_train_reg)
linear_reg_time = time.time() - start_time

linear_preds = linear_model.predict(X_test_reg)

linear_results = {
    "MAE": mean_absolute_error(y_test_reg, linear_preds),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, linear_preds)),
    "R2": r2_score(y_test_reg, linear_preds),
    "Training Time Seconds": linear_reg_time
}

linear_results


In [ ]:
linear_coefficients = pd.DataFrame({
    "Feature": regression_features,
    "Coefficient": linear_model.coef_
}).sort_values(by="Coefficient", ascending=False)

linear_coefficients


In [ ]:
sns.barplot(data=linear_coefficients, x="Coefficient", y="Feature")
plt.title("Linear Regression Coefficients for Dental Loss Ratio")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.show()


## 9. Random Forest Regression for Dental Loss Ratio

In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=None
)

start_time = time.time()
rf_reg.fit(X_train_reg, y_train_reg)
rf_reg_time = time.time() - start_time

rf_preds = rf_reg.predict(X_test_reg)

rf_reg_results = {
    "MAE": mean_absolute_error(y_test_reg, rf_preds),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, rf_preds)),
    "R2": r2_score(y_test_reg, rf_preds),
    "Training Time Seconds": rf_reg_time
}

rf_reg_results


In [ ]:
rf_importance = pd.DataFrame({
    "Feature": regression_features,
    "Importance": rf_reg.feature_importances_
}).sort_values(by="Importance", ascending=False)

sns.barplot(data=rf_importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance for Dental Loss Ratio")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

rf_importance


## 10. Classification: Predicting Business Type

In [ ]:
# Classification target: Business Type

if not business_col:
    print("Business Type column not found. Skipping classification.")
else:
    classification_features = [
        premium_col,
        payment_col,
        members_col,
        member_months_col,
        loss_ratio_col,
        avg_premium_col
    ]
    classification_features = [col for col in classification_features if col and col in data.columns]

    class_df = data[classification_features + [business_col]].dropna(subset=[business_col]).copy()

    # Remove very rare classes for stable modeling
    class_counts = class_df[business_col].value_counts()
    valid_classes = class_counts[class_counts >= 2].index
    class_df = class_df[class_df[business_col].isin(valid_classes)]

    print("Classification features:", classification_features)
    print("Class distribution:")
    print(class_df[business_col].value_counts())


In [ ]:
if business_col:
    X_cls = class_df[classification_features]
    y_cls = class_df[business_col]

    X_cls = SimpleImputer(strategy="median").fit_transform(X_cls)

    le_target = LabelEncoder()
    y_cls_encoded = le_target.fit_transform(y_cls)

    stratify_value = y_cls_encoded if min(np.bincount(y_cls_encoded)) >= 2 else None

    X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
        X_cls,
        y_cls_encoded,
        test_size=0.30,
        random_state=42,
        stratify=stratify_value
    )

    rf_cls = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    )

    start_time = time.time()
    rf_cls.fit(X_train_cls, y_train_cls)
    rf_cls_time = time.time() - start_time

    cls_preds = rf_cls.predict(X_test_cls)

    print("Random Forest Classification Accuracy:", round(accuracy_score(y_test_cls, cls_preds), 4))
    print("Training Time Seconds:", round(rf_cls_time, 4))
    print("\nClassification Report:")
    print(classification_report(y_test_cls, cls_preds, target_names=le_target.classes_))
else:
    print("Skipping classification.")


In [ ]:
if business_col:
    cls_importance = pd.DataFrame({
        "Feature": classification_features,
        "Importance": rf_cls.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    sns.barplot(data=cls_importance, x="Importance", y="Feature")
    plt.title("Feature Importance for Business Type Prediction")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()

    cls_importance
else:
    print("Skipping classification feature importance.")


## 11. Profiling Data Loading and Modeling

In [ ]:
def profiling_workflow():
    temp_data = pd.read_csv(file_path)
    temp_data.columns = (
        temp_data.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("/", "_")
        .str.replace("-", "_")
        .str.replace("__", "_")
    )
    return temp_data.shape

profiler = cProfile.Profile()
profiler.enable()

profile_result = profiling_workflow()

profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("cumulative")
stats.print_stats(15)

print("Profile result:", profile_result)
print(stream.getvalue())


## 12. Performance Summary

In [ ]:
performance_summary = pd.DataFrame([
    {
        "Task": "Data Loading",
        "Method": "Pandas read_csv",
        "Time Seconds": pandas_load_time
    },
    {
        "Task": "Data Loading",
        "Method": "Dask read_csv",
        "Time Seconds": dask_load_time
    },
    {
        "Task": "Regression",
        "Method": "Linear Regression",
        "Time Seconds": linear_reg_time
    },
    {
        "Task": "Regression",
        "Method": "Random Forest Regression",
        "Time Seconds": rf_reg_time
    }
])

if business_col:
    performance_summary = pd.concat([
        performance_summary,
        pd.DataFrame([{
            "Task": "Classification",
            "Method": "Random Forest Classifier",
            "Time Seconds": rf_cls_time
        }])
    ], ignore_index=True)

performance_summary


In [ ]:
sns.barplot(data=performance_summary.dropna(), x="Time Seconds", y="Method", hue="Task")
plt.title("Performance Comparison")
plt.xlabel("Time in Seconds")
plt.ylabel("Method")
plt.show()


## 13. Save Cleaned Data and Output Files

In [ ]:
output_dir = Path("dental_loss_ratio_outputs")
output_dir.mkdir(exist_ok=True)

data.to_csv(output_dir / "cleaned_dental_loss_ratio_data.csv", index=False)
performance_summary.to_csv(output_dir / "performance_summary.csv", index=False)
linear_coefficients.to_csv(output_dir / "linear_regression_coefficients.csv", index=False)
rf_importance.to_csv(output_dir / "rf_regression_feature_importance.csv", index=False)

if business_col:
    cls_importance.to_csv(output_dir / "business_type_feature_importance.csv", index=False)

print("Saved output files in:", output_dir)
for file in output_dir.iterdir():
    print("-", file)


## 14. Key Findings

- Dental Loss Ratio can be analyzed using financial features such as dental payments, premiums, members, and member months.
- Data loading performance can be compared across Pandas and Dask for scalability analysis.
- Regression models help estimate Dental Loss Ratio based on insurer financial/customer metrics.
- Random Forest feature importance helps identify the most influential predictors.
- Profiling helps identify runtime bottlenecks in data loading and modeling workflows.


## 15. Conclusion

This project demonstrates an insurance analytics workflow using Washington State dental loss ratio data. It combines data cleaning, exploratory analysis, performance benchmarking, regression modeling, classification modeling, and profiling.

The workflow supports insurance data analysis by identifying trends in premiums, payments, member coverage, and dental loss ratios while also comparing computational efficiency across data processing methods.
